In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

Algorithm: K-Means clustering (unsupervised) to group research centers into three quality tiers (Premium, Standard, Basic) based on infrastructure and healthcare access features.

In [ ]:
filename = 'research_centers.csv'

# Load

In [ ]:
raw_df = pd.read_csv(filename)
raw_df.head()

In [ ]:
raw_df.info(), raw_df.duplicated().sum()

# Explore

In [ ]:
raw_df['researchCenterName'].nunique(), raw_df['city'].nunique()

In [ ]:
city_counts = raw_df.groupby('city')['researchCenterId'].nunique().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.tab20.colors  # or `tab10` etc
ax = city_counts.plot(
    kind='barh',
    legend=False,
    ax=ax,
    grid=True,
    color=[colors[i % len(colors)] for i in range(len(city_counts))],
    width=0.8
)

for i, value in enumerate(city_counts):
    ax.text(value + 0.1, i, str(value), va='center')

ax.set_title('Unique Research Centers per City')
ax.set_xlabel('Unique researchCenterId count')
ax.set_ylabel('City')
plt.tight_layout()
plt.show()

In [ ]:
city_facilities_df = raw_df.groupby('city')['internalFacilitiesCount'].agg(['mean', 'min', 'max', 'sum', 'count', 'nunique'])
max_bin_count = int(city_facilities_df['max'].max())
city_facilities_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

raw_df['internalFacilitiesCount'].hist(bins=max_bin_count, ax=ax, grid=True)
ax.set_title(f'Distribution of Internal Facilities Count - Total is {raw_df["internalFacilitiesCount"].sum()}')
ax.set_xlabel('Internal Facilities Count')
ax.set_ylabel('Frequency')
ax.set_xticks(range(raw_df['internalFacilitiesCount'].min(), raw_df['internalFacilitiesCount'].max() + 1))
ax.set_xlim(raw_df['internalFacilitiesCount'].min() - 0.5, raw_df['internalFacilitiesCount'].max() + 0.5)
plt.show()

In [ ]:
cities = raw_df['city'].sort_values().unique()

fig, axes = plt.subplots(3, 2, figsize=(12, 10), sharex=False)
axes = axes.flatten()

for i, city in enumerate(cities):
    subset = raw_df.loc[raw_df['city'] == city, 'internalFacilitiesCount']
    subset.plot(
        kind='hist',
        bins=max_bin_count,
        ax=axes[i],
        edgecolor='black',
        alpha=0.7,
        title=city
    )
    axes[i].set_xlabel('internalFacilitiesCount')
    axes[i].set_ylabel('Frequency')

for j in range(len(cities), len(axes)):
    axes[j].axis('off')

fig.suptitle('internalFacilitiesCount Distribution by City (3x2 grid)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()